<a href="https://colab.research.google.com/github/ebubesimeon82-bit/CodeAlpha_.ipynb/blob/main/CodeAlpha_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
CodeAlpha Machine Learning Internship — Task 1: Credit Scoring Model
=====================================================================
Predicts whether an individual will experience serious delinquency in the
next 2 years (`dlq_2yrs`) using their financial history.

Pipeline: EDA -> preprocessing -> feature engineering -> model training
(Logistic Regression, Random Forest, Gradient Boosting) -> evaluation.

Run:
    python credit_scoring_model.py
Outputs:
    - Printed EDA summary and metrics for each model
    - eda_target_balance.png, correlation_heatmap.png
    - roc_curves.png, confusion_matrices.png, feature_importance.png
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

sns.set_style("whitegrid")
RANDOM_STATE = 42
# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
DATA_PATH = "Credit Risk Benchmark Dataset.csv"
df = pd.read_csv(DATA_PATH)
TARGET = "dlq_2yrs"

print("=" * 70)
print("DATA OVERVIEW")
print("=" * 70)
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nTarget balance:\n{df[TARGET].value_counts(normalize=True)}")

# ---------------------------------------------------------------------------
# 2. EDA plots
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df[TARGET].value_counts().plot(
    kind="bar", ax=axes[0], color=["#2c7fb8", "#d95f0e"]
)
axes[0].set_title("Target Class Balance (dlq_2yrs)")
axes[0].set_xticklabels(["No Delinquency (0)", "Delinquency (1)"], rotation=0)
axes[0].set_ylabel("Count")

sns.boxplot(x=TARGET, y="rev_util", data=df[df["rev_util"] < 5], ax=axes[1])
axes[1].set_title("Revolving Utilization by Class (outliers capped for plot)")

plt.tight_layout()
plt.savefig("eda_target_balance.png", dpi=150)
plt.close()

plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.close()

print("\nSaved EDA plots: eda_target_balance.png, correlation_heatmap.png")

# ---------------------------------------------------------------------------
# 3. Preprocessing
# ---------------------------------------------------------------------------
# rev_util and debt_ratio contain extreme outliers (data-entry errors, e.g.
# utilization > 1 should be rare/impossible) — cap at the 99th percentile
# instead of dropping rows, to keep the dataset size intact.
df_clean = df.copy()
for col in ["rev_util", "debt_ratio", "monthly_inc"]:
    cap = df_clean[col].quantile(0.99)
    df_clean[col] = np.where(df_clean[col] > cap, cap, df_clean[col])

# ---------------------------------------------------------------------------
# 4. Feature engineering
# ---------------------------------------------------------------------------
df_clean["total_late_payments"] = (
    df_clean["late_30_59"] + df_clean["late_60_89"] + df_clean["late_90"]
)
df_clean["income_per_dependent"] = df_clean["monthly_inc"] / (
    df_clean["dependents"] + 1
)
df_clean["credit_lines_per_age"] = df_clean["open_credit"] / df_clean["age"]

feature_cols = [c for c in df_clean.columns if c != TARGET]
X = df_clean[feature_cols]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------------------------
# 5. Train models
# ---------------------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        random_state=RANDOM_STATE
    ),
}

results = {}
roc_data = {}

print("\n" + "=" * 70)
print("MODEL TRAINING & EVALUATION")
print("=" * 70)

for name, model in models.items():
    # Logistic Regression needs scaled features; tree models don't.
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        "Accuracy": acc, "Precision": prec, "Recall": rec,
        "F1-Score": f1, "ROC-AUC": auc
    }
    roc_data[name] = roc_curve(y_test, y_proba)

    print(f"\n--- {name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")

# ---------------------------------------------------------------------------
# 6. Comparison table
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results).T.round(4)
print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print(results_df)
results_df.to_csv("model_comparison_results.csv")

# ---------------------------------------------------------------------------
# 7. ROC curves
# ---------------------------------------------------------------------------
plt.figure(figsize=(7, 6))
for name, (fpr, tpr, _) in roc_data.items():
    auc = results[name]["ROC-AUC"]
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Credit Scoring Models")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)
plt.close()

# ---------------------------------------------------------------------------
# 8. Confusion matrices
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, model) in zip(axes, models.items()):
    if name == "Logistic Regression":
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.close()

# ---------------------------------------------------------------------------
# 9. Feature importance (Random Forest)
# ---------------------------------------------------------------------------
rf_model = models["Random Forest"]
importances = pd.Series(
    rf_model.feature_importances_, index=feature_cols
).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.plot(kind="barh", color="#2c7fb8")
plt.gca().invert_yaxis()
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.close()

print("\nSaved: roc_curves.png, confusion_matrices.png, feature_importance.png")
print("Saved: model_comparison_results.csv")

best_model = results_df["ROC-AUC"].idxmax()
print(f"\nBest performing model by ROC-AUC: {best_model}")


DATA OVERVIEW
Shape: (16714, 11)

Missing values:
rev_util       0
age            0
late_30_59     0
debt_ratio     0
monthly_inc    0
open_credit    0
late_90        0
real_estate    0
late_60_89     0
dependents     0
dlq_2yrs       0
dtype: int64

Target balance:
dlq_2yrs
0    0.5
1    0.5
Name: proportion, dtype: float64

Saved EDA plots: eda_target_balance.png, correlation_heatmap.png

MODEL TRAINING & EVALUATION

--- Logistic Regression ---
Accuracy:  0.7568
Precision: 0.7597
Recall:    0.7510
F1-Score:  0.7553
ROC-AUC:   0.8386

--- Random Forest ---
Accuracy:  0.7789
Precision: 0.7942
Recall:    0.7528
F1-Score:  0.7730
ROC-AUC:   0.8619

--- Gradient Boosting ---
Accuracy:  0.7780
Precision: 0.7844
Recall:    0.7666
F1-Score:  0.7754
ROC-AUC:   0.8607

MODEL COMPARISON
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression    0.7568     0.7597  0.7510    0.7553   0.8386
Random Forest          0.7789     0.7942  0.7528    0.7730   0.8619
Gradien